# AgenticRAG-Bench — Week 3 (Second Attempt)
## Fixed: recursion_limit, OMP crash, context explosion, timeout

**Fixes applied from first attempt:**
1. `KMP_DUPLICATE_LIB_OK=TRUE` — fixes FAISS/OMP double-init crash
2. `recursion_limit=28` — each LangGraph "recursion" = 1 node (LLM or tool), need 2×max_tools+buffer
3. Truncated tool output to 500 chars/doc — prevents context explosion
4. 120s per-question timeout — no more infinite hangs
5. Removed duplicate system prompt (was in both `create_react_agent` and `invoke`)
6. Incremental save after each question

**What does NOT change from Week 2:**
- Agent: LangGraph ReAct + Llama 3.1 8B
- Embeddings: nomic-embed-text
- k = 3 | system prompt ON | max_steps = 10

---

## Cell 0 — Environment + imports

In [25]:
import sys
import os
import json
import time
import math
import re
import signal
from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
os.environ["RAGAS_DO_NOT_PARALLELIZE"] = "true"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Fix OMP libomp double-init crash with FAISS
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

import requests
r = requests.get("http://localhost:11434/api/tags", timeout=3)
models = [m['name'] for m in r.json().get('models', [])]
print("Ollama models:", models)

from datasets import load_dataset
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent

Path(PROJECT_ROOT / "data/questions").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "data/trajectories").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "notes").mkdir(parents=True, exist_ok=True)

print("All imports OK")

Project root: /Users/idhantsingh/Desktop/agenticrag-bench
Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
Ollama models: ['nomic-embed-text:latest', 'llama3.1:8b']
All imports OK


---
## Cell 1 — All metric classes (D1, D2, D3, D5)

All four metrics in one cell. Unchanged from Week 3 attempt 1.

In [26]:
# ── D1: Answer Correctness ───────────────────────────────────────

class D1_AnswerCorrectness:
    name = "D1_answer_correctness"
    STOPWORDS = {
        "the","a","an","is","was","of","in","to","and","or",
        "it","its","be","been","being","have","has","had",
        "by","at","from","with","that","this","are"
    }
    LEGAL_SUFFIXES = {
        "inc","incorporated","corp","corporation",
        "ltd","limited","llc","plc","gmbh","company"
    }

    def _normalize(self, text):
        text = text.lower().strip()
        text = re.sub(r'[^\w\s]', ' ', text)
        words = [w for w in text.split() if w not in self.LEGAL_SUFFIXES]
        return ' '.join(words)

    def _meaningful_words(self, text):
        text = re.sub(r'[^\w\s]', ' ', text)
        return {w for w in text.split()
                if w not in self.STOPWORDS
                and w not in self.LEGAL_SUFFIXES
                and len(w) > 1}

    def _is_degenerate(self, predicted):
        """Detect degenerate outputs: raw tool dumps, refusals, empty."""
        p = predicted.strip()
        return (
            (p.startswith('{') and 'search_knowledge_base' in p)
            or p.lower().startswith('sorry')
            or len(p) < 5
        )

    def score(self, predicted, ground_truth):
        degenerate = self._is_degenerate(predicted)
        if degenerate:
            return {
                "exact_match": False, "near_match": False,
                "token_recall": 0.0, "seq_sim": 0.0,
                "score": 0.0, "degenerate_output": True
            }

        pred_raw  = predicted.lower().strip()
        truth_raw = ground_truth.lower().strip()
        exact_raw  = truth_raw in pred_raw
        pred_norm  = self._normalize(predicted)
        truth_norm = self._normalize(ground_truth)
        if ' ' not in truth_norm:
            exact_norm = (
                pred_norm == truth_norm or
                pred_norm.startswith(truth_norm + ' ') or
                (' is ' + truth_norm) in pred_norm or
                (' was ' + truth_norm) in pred_norm
            )
        else:
            exact_norm = truth_norm in pred_norm
        exact_match = exact_raw or exact_norm
        pred_words  = self._meaningful_words(pred_norm)
        truth_words = self._meaningful_words(truth_norm)
        if truth_words:
            recall = len(pred_words & truth_words) / len(truth_words)
        else:
            recall = 1.0 if exact_match else 0.0
        seq = SequenceMatcher(None, truth_norm,
                              pred_norm[:len(truth_norm)*3]).ratio()
        near = (recall >= 0.6) or (seq >= 0.7 and recall >= 0.4)
        if exact_match:    score = 1.0
        elif near:         score = round(0.5 + recall * 0.5, 3)
        else:
            intersection = pred_words & truth_words
            union        = pred_words | truth_words
            score = round(len(intersection)/len(union), 3) if union else 0.0
        return {
            "exact_match": exact_match, "near_match": near,
            "token_recall": round(recall,3), "seq_sim": round(seq,3),
            "score": score, "degenerate_output": False
        }




# ── D2: Retrieval Step Quality ───────────────────────────────────

class D2_RetrievalStepQuality:
    name = "D2_retrieval_step_quality"
    def _doc_is_relevant(self, doc_text, supporting_paragraphs):
        doc_lower = doc_text.lower()
        for sup in supporting_paragraphs:
            if sup[:80].lower() in doc_lower:
                return True
            sup_words = set(sup.lower().split())
            doc_words = set(doc_lower.split())
            if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.7:
                return True
        return False

    def score(self, trajectory_steps, supporting_paragraphs):
        if not trajectory_steps:
            return {"avg_precision_at_k":0.0, "best_step_precision":0.0,
                    "any_relevant_found":False, "step_precisions":[], "d2_score":0.0}
        step_precisions = []
        any_relevant = False
        for step in trajectory_steps:
            docs = step.get("retrieved_content", [])
            if not docs:
                step_precisions.append(0.0)
                continue
            hits = sum(1 for doc in docs if self._doc_is_relevant(doc, supporting_paragraphs))
            prec = hits / len(docs)
            step_precisions.append(round(prec, 3))
            if hits > 0: any_relevant = True
        avg_prec = sum(step_precisions) / len(step_precisions)
        best_prec = max(step_precisions)
        d2_score = round(min(1.0, avg_prec * 0.8 + (0.2 if any_relevant else 0.0)), 3)
        return {"avg_precision_at_k": round(avg_prec,3), "best_step_precision": round(best_prec,3),
                "any_relevant_found": any_relevant, "step_precisions": step_precisions, "d2_score": d2_score}




# ── D3: Planning Coherence ───────────────────────────────────────

class D3_PlanningCoherence:
    name = "D3_planning_coherence"
    def _query_diversity(self, queries):
        if len(queries) <= 1: return 0.0
        unique = list(set(queries))
        if len(unique) == 1: return 0.0
        pairs = []
        for i in range(len(unique)):
            for j in range(i+1, len(unique)):
                pairs.append(1.0 - SequenceMatcher(None, unique[i], unique[j]).ratio())
        return round(sum(pairs)/len(pairs), 3) if pairs else 0.0

    def score(self, trajectory_summary, num_hops):
        total_steps    = trajectory_summary.get("total_steps", 0)
        unique_queries = trajectory_summary.get("unique_queries", [])
        redundancy     = trajectory_summary.get("redundancy_rate", 0)
        loop_detected  = trajectory_summary.get("loop_detected", False)
        num_unique = len(unique_queries)
        hop_coverage = min(1.0, num_unique / max(num_hops, 1))
        diversity = self._query_diversity(unique_queries)
        loop_penalty      = 0.4 if loop_detected else 0.0
        undershoot        = max(0.0, (num_hops - num_unique) / max(num_hops, 1))
        undershoot_penalty = undershoot * 0.5
        raw = (hop_coverage * 0.5) + (diversity * 0.5) - loop_penalty - undershoot_penalty
        d3_score = round(max(0.0, min(1.0, raw)), 3)
        return {"num_hops_required": num_hops, "unique_queries_made": num_unique,
                "hop_coverage": round(hop_coverage,3), "query_diversity": round(diversity,3),
                "loop_penalty": loop_penalty, "undershoot_penalty": round(undershoot_penalty,3),
                "d3_score": d3_score}




# ── D5: Trajectory Efficiency ────────────────────────────────────

class D5_TrajectoryEfficiency:
    name = "D5_trajectory_efficiency"
    BASELINE_TOKENS_PER_STEP = 400
    def score(self, trajectory_summary, answer_correct, degenerate=False):
        if degenerate:
            return {"total_steps": 0, "redundancy_rate": 0,
                    "tokens_per_step": 0, "loop_detected": False,
                    "loop_penalty_applied": 0, "efficiency_score": 0.0,
                    "total_tokens": 0, "tokens_per_correct_answer": None,
                    "degenerate": True}

        total_steps    = trajectory_summary.get("total_steps", 0)
        redundancy     = trajectory_summary.get("redundancy_rate", 0)
        tokens_per_step= trajectory_summary.get("tokens_per_step", 0)
        loop_detected  = trajectory_summary.get("loop_detected", False)
        total_tokens   = trajectory_summary.get("total_tokens", 0)
        token_penalty  = min(1.0, tokens_per_step / (self.BASELINE_TOKENS_PER_STEP * 2))
        loop_penalty   = 0.3 if loop_detected else 0.0
        raw  = 1.0 - (redundancy*0.5) - (token_penalty*0.3) - loop_penalty
        eff  = max(0.0, round(raw, 3))
        return {"total_steps": total_steps, "redundancy_rate": redundancy,
                "tokens_per_step": tokens_per_step, "loop_detected": loop_detected,
                "loop_penalty_applied": loop_penalty, "efficiency_score": eff,
                "total_tokens": total_tokens,
                "tokens_per_correct_answer": total_tokens if answer_correct else None}


# ── Quick tests ──────────────────────────────────────────────────

d1 = D1_AnswerCorrectness()
d2 = D2_RetrievalStepQuality()
d3 = D3_PlanningCoherence()
d5 = D5_TrajectoryEfficiency()

# D1 tests
assert d1.score("Bombardier Aerospace made it", "Bombardier Inc.")["near_match"] == True
assert d1.score("Miquette Giraudy is her name", "Miquette Giraudy")["exact_match"] == True
assert d1.score("I don't know", "Paris")["score"] == 0.0
assert d1.score("Textron Aviation acquired Bombardier", "Bombardier Inc.")["exact_match"] == False,     "Bug: Textron answer should not match Bombardier Inc."
print("D1 tests passed")

# D2 test
fake_steps = [
    {"retrieved_content": ["Steve Hillage is a British musician and spouse of Miquette Giraudy",
                            "The Godfather was directed by Coppola",
                            "Churchill was Prime Minister"]},
]
d2_result = d2.score(fake_steps, ["Steve Hillage is a British musician and spouse of Miquette Giraudy"])
assert d2_result["any_relevant_found"] == True
assert d2_result["avg_precision_at_k"] == round(1/3, 3)
print("D2 tests passed")

# D3 test
traj_good = {"total_steps":2, "unique_queries":["Steve Hillage musician","Steve Hillage spouse"],
             "redundancy_rate":0.0, "loop_detected":False}
traj_bad  = {"total_steps":2, "unique_queries":["Green performer spouse"],
             "redundancy_rate":0.5, "loop_detected":True}
d3_good = d3.score(traj_good, num_hops=2)
d3_bad  = d3.score(traj_bad,  num_hops=2)
assert d3_good["d3_score"] > d3_bad["d3_score"], "Good planning should score higher"
print(f"D3 tests passed — good={d3_good['d3_score']}, bad={d3_bad['d3_score']}")

print("\nAll metric classes ready")

D1 tests passed
D2 tests passed
D3 tests passed — good=0.619, bad=0.0

All metric classes ready


---
## Cell 2 — TrajectoryLogger (same as Week 2)

In [27]:

class TrajectoryLogger:
    def __init__(self, max_steps=10):
        self.max_steps = max_steps
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def reset(self):
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def log(self, query, retrieved_docs, latency_ms, supporting_paragraphs=None):
        is_loop = bool(self.steps and self.steps[-1]['query'] == query)
        if is_loop:
            self.loop_detected = True
            self.loop_query = query
        precision_at_k = 0.0
        if supporting_paragraphs and retrieved_docs:
            def _is_relevant(doc):
                doc_words = set(doc.lower().split())
                for sup in supporting_paragraphs:
                    sup_words = set(sup.lower().split())
                    if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.5:
                        return True
                return False
            hits = sum(1 for doc in retrieved_docs if _is_relevant(doc))
            precision_at_k = round(hits / len(retrieved_docs), 3)
        tokens_estimate = (len(query) + sum(len(d) for d in retrieved_docs)) // 4
        step = {
            "step_id":            len(self.steps) + 1,
            "query":              query,
            "num_docs_retrieved": len(retrieved_docs),
            "retrieved_content":  retrieved_docs,
            "precision_at_k":     precision_at_k,
            "is_loop_step":       is_loop,
            "tokens_estimate":    tokens_estimate,
            "latency_ms":         round(latency_ms, 1)
        }
        self.steps.append(step)
        return step

    def at_limit(self):
        return len(self.steps) >= self.max_steps

    def summary(self):
        if not self.steps: return {}
        n = len(self.steps)
        unique = list(set(s['query'] for s in self.steps))
        redundant = n - len(unique)
        total_tokens = sum(s['tokens_estimate'] for s in self.steps)
        avg_prec = sum(s['precision_at_k'] for s in self.steps) / n
        return {
            "total_steps": n, "unique_queries": unique,
            "redundant_queries": redundant,
            "redundancy_rate": round(redundant / n, 3),
            "loop_detected": self.loop_detected,
            "loop_steps": sum(1 for s in self.steps if s['is_loop_step']),
            "avg_step_precision_at_k": round(avg_prec, 3),
            "total_tokens": total_tokens,
            "total_latency_ms": round(sum(s['latency_ms'] for s in self.steps), 1),
            "tokens_per_step": round(total_tokens / n, 1)
        }
print('TrajectoryLogger ready')


TrajectoryLogger ready


---
## Cell 3 — Load 50 questions with difficulty labels

In [28]:
print("Loading MuSiQue dataset...")

dataset = None
for name, config in [
    ("google-research-datasets/musique", "musique_ans_v1.0"),
    ("dgslibisey/MuSiQue", None),
]:
    try:
        dataset = load_dataset(name, config) if config else load_dataset(name)
        print(f"Loaded from: {name}")
        break
    except Exception as e:
        print(f"Failed: {name}")

if dataset is None:
    raise ValueError("Cannot load MuSiQue")


def assign_difficulty_labels(item):
    """
    Assign 4-axis difficulty labels to a MuSiQue item.
    Based on Enterprise RAG paper taxonomy (Narita et al. 2026).
    """
    num_hops = len(item['question_decomposition'])
    num_distractors = sum(1 for p in item['paragraphs'] if not p['is_supporting'])
    num_supporting  = sum(1 for p in item['paragraphs'] if p['is_supporting'])

    # Axis 1: Reasoning complexity (1=single hop, 2=two hop, 3=three+ hop)
    reasoning_complexity = min(3, num_hops)

    # Axis 2: Retrieval difficulty (1=easy, 2=medium, 3=hard)
    # Based on ratio of distractors to supporting docs
    distractor_ratio = num_distractors / max(num_supporting, 1)
    if distractor_ratio <= 3:   retrieval_difficulty = 1
    elif distractor_ratio <= 7: retrieval_difficulty = 2
    else:                       retrieval_difficulty = 3

    # Axis 3: Source structure (text = all plain text, mixed = has lists/dates)
    # Heuristic: if answer is a number or year, likely requires structured parsing
    answer = item['answer'].lower()
    is_structured = (answer.isdigit() or
                     any(c.isdigit() for c in answer) and len(answer) <= 6)
    source_structure = "mixed" if is_structured else "text"

    # Axis 4: Explainability required (does question ask why/how)
    q_lower = item['question'].lower()
    explainability_required = any(
        q_lower.startswith(w) for w in ["why", "how", "explain", "describe"]
    )

    # D6 tag: multi-axis difficulty (True when 2+ axes are high)
    high_axes = sum([
        reasoning_complexity >= 2,
        retrieval_difficulty >= 2,
        source_structure == "mixed",
        explainability_required
    ])
    multi_axis_hard = high_axes >= 2

    return {
        "reasoning_complexity":  reasoning_complexity,
        "retrieval_difficulty":  retrieval_difficulty,
        "source_structure":      source_structure,
        "explainability_required": explainability_required,
        "num_high_difficulty_axes": high_axes,
        "multi_axis_hard":       multi_axis_hard   # key for D6 analysis
    }


# Load 50 questions
questions = []
for i, item in enumerate(dataset['validation']):
    if i >= 50:
        break

    supporting = [p['paragraph_text'] for p in item['paragraphs'] if p['is_supporting']]
    distractors= [p['paragraph_text'] for p in item['paragraphs'] if not p['is_supporting']]
    all_paras  = [p['paragraph_text'] for p in item['paragraphs']]
    difficulty = assign_difficulty_labels(item)

    questions.append({
        "id":                    item['id'],
        "question":              item['question'],
        "answer":                item['answer'],
        "num_hops":              len(item['question_decomposition']),
        "decomposition":         [{"sub_q":d['question'],"sub_a":d['answer']}
                                   for d in item['question_decomposition']],
        "paragraphs":            all_paras,
        "supporting_paragraphs": supporting,
        "distractor_paragraphs": distractors,
        "num_supporting":        len(supporting),
        "num_distractors":       len(distractors),
        "difficulty":            difficulty
    })

# Save
out_path = PROJECT_ROOT / "data/questions/3_musique_50.json"
with open(out_path, "w") as f:
    json.dump(questions, f, indent=2)

print(f"Saved {len(questions)} questions to {out_path}")
print()

# Dataset statistics
hop_counts  = {1:0, 2:0, 3:0}
multi_axis  = sum(1 for q in questions if q['difficulty']['multi_axis_hard'])
rc_dist     = {1:0, 2:0, 3:0}
rd_dist     = {1:0, 2:0, 3:0}

for q in questions:
    hops = min(3, q['num_hops'])
    hop_counts[hops] = hop_counts.get(hops, 0) + 1
    rc = q['difficulty']['reasoning_complexity']
    rd = q['difficulty']['retrieval_difficulty']
    rc_dist[rc] = rc_dist.get(rc, 0) + 1
    rd_dist[rd] = rd_dist.get(rd, 0) + 1

print("Dataset v1 statistics (50 questions):")
print(f"  Hop distribution:            {hop_counts}")
print(f"  Reasoning complexity 1/2/3:  {rc_dist}")
print(f"  Retrieval difficulty 1/2/3:  {rd_dist}")
print(f"  Multi-axis hard questions:   {multi_axis}/50 = {multi_axis/50:.0%}")
print(f"  Avg KB size per question:    {sum(len(q['paragraphs']) for q in questions)//len(questions)} docs")
print(f"  Avg supporting docs:         {sum(q['num_supporting'] for q in questions)/len(questions):.1f}")
print(f"  Avg distractors:             {sum(q['num_distractors'] for q in questions)/len(questions):.1f}")

Loading MuSiQue dataset...
Failed: google-research-datasets/musique
Loaded from: dgslibisey/MuSiQue
Saved 50 questions to /Users/idhantsingh/Desktop/agenticrag-bench/data/questions/3_musique_50.json

Dataset v1 statistics (50 questions):
  Hop distribution:            {1: 0, 2: 50, 3: 0}
  Reasoning complexity 1/2/3:  {1: 0, 2: 50, 3: 0}
  Retrieval difficulty 1/2/3:  {1: 0, 2: 0, 3: 50}
  Multi-axis hard questions:   50/50 = 100%
  Avg KB size per question:    20 docs
  Avg supporting docs:         2.0
  Avg distractors:             18.0


---
## Cell 4 — Build agent (FIXED: truncated output, system prompt once)

In [29]:
print("Loading embedding model...")
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
_ = embedding_model.embed_query("test")
print("nomic-embed-text OK")

TRAJECTORY_LOGGER = TrajectoryLogger(max_steps=10)
current_context = {"store": None, "supporting_paragraphs": []}

# FIX 1: Truncate tool output to keep LLM context small
MAX_CHARS_PER_DOC = 500   # was unlimited → context explodes

@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for information relevant to the query.
    For multi-hop questions ALWAYS search at least twice:
    - First: find what the subject IS
    - Second: find the specific fact about that subject
    Never repeat the same query. If results are unhelpful, change the query.
    """
    if TRAJECTORY_LOGGER.at_limit():
        return "[SEARCH LIMIT REACHED: provide your best answer now.]"

    # FIX: Query Rewriting to improve D2
    rewrite_prompt = f"Rewrite this search query to be highly effective for a vector database. Focus on keywords and entities. Remove conversational filler. Output ONLY the rewritten query text: '{query}'"
    try:
        rewritten_query = llm.invoke([('user', rewrite_prompt)]).content.strip()
    except Exception:
        rewritten_query = query

    t0 = time.time()
    results = current_context["store"].similarity_search(rewritten_query, k=3)
    latency = (time.time() - t0) * 1000
    texts = [r.page_content for r in results]

    step = TRAJECTORY_LOGGER.log(f"{query} [Rewritten as: {rewritten_query}]", texts, latency,
                                  current_context["supporting_paragraphs"])

    # Truncate for LLM context — full text is already saved in trajectory
    truncated = [t[:MAX_CHARS_PER_DOC] for t in texts]

    warning = ""
    if step["is_loop_step"]:
        warning = "\n\n[You already searched this query. Use a DIFFERENT query.]"

    # FIX: Nudge agent to keep searching after first step
    nudge = ""
    if len(TRAJECTORY_LOGGER.steps) == 1:
        nudge = "\n\n[NOTE: You have completed 1 search. Multi-hop questions require at least 2 searches. Search again with a DIFFERENT query before answering.]"

    return "\n\n---\n\n".join(truncated) + warning + nudge


SYSTEM_PROMPT = """You are a research assistant answering multi-hop questions.
These questions require combining facts from 2+ sources.

RULES:
1. ALWAYS search at least twice before answering.
2. First search: find what the main entity IS.
3. Second search: find the specific fact about that entity.
4. Never repeat a query — vary your searches.
5. Answer concisely after at least 2 searches."""

llm   = ChatOllama(model="llama3.1:8b", temperature=0)
agent = create_react_agent(llm, [search_knowledge_base], prompt=SYSTEM_PROMPT)

print("Agent ready — k=3, system prompt ON, max_steps=10")

Loading embedding model...
nomic-embed-text OK
Agent ready — k=3, system prompt ON, max_steps=10


/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_94721/1371913788.py:65: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [search_knowledge_base], prompt=SYSTEM_PROMPT)


---
## Cell 5 — Run function with timeout + recursion_limit

In [30]:
d1_metric = D1_AnswerCorrectness()
d2_metric = D2_RetrievalStepQuality()
d3_metric = D3_PlanningCoherence()
d5_metric = D5_TrajectoryEfficiency()

# FIX 2: Timeout wrapper
class QuestionTimeoutError(Exception):
    pass

def _timeout_handler(signum, frame):
    raise QuestionTimeoutError("Question timed out")

QUESTION_TIMEOUT = 120  # seconds per question


def run_question(q):
    global current_context
    TRAJECTORY_LOGGER.reset()

    # Build per-question FAISS store
    docs  = [Document(page_content=p) for p in q['paragraphs']]
    store = FAISS.from_documents(docs, embedding_model)
    current_context["store"]                = store
    current_context["supporting_paragraphs"]= q['supporting_paragraphs']

    t0 = time.time()

    # FIX 2: timeout so one question can't hang forever
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(QUESTION_TIMEOUT)

    try:
        # FIX 3: recursion_limit=28 (each LangGraph "recursion" = 1 node)
        # Need 2*max_tool_calls + buffer for final answer
        result  = agent.invoke(
            {"messages": [("user", q["question"])]},
            config={"recursion_limit": 28}
        )
        answer  = result["messages"][-1].content
        success = True
    except QuestionTimeoutError:
        answer  = "TIMEOUT"
        success = False
        print(f"    ⚠ TIMEOUT after {QUESTION_TIMEOUT}s")
    except Exception as e:
        answer  = f"ERROR: {e}"
        success = False
        print(f"    ⚠ ERROR: {e}")
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    wall_ms   = (time.time() - t0) * 1000
    traj_sum  = TRAJECTORY_LOGGER.summary()

    d1 = d1_metric.score(answer, q["answer"])
    d2 = d2_metric.score(TRAJECTORY_LOGGER.steps, q["supporting_paragraphs"])
    d3 = d3_metric.score(traj_sum, q["num_hops"])
    d5 = d5_metric.score(traj_sum, d1["exact_match"], degenerate=d1.get("degenerate_output", False))

    return {
        "question_id":        q["id"],
        "question":           q["question"],
        "ground_truth":       q["answer"],
        "num_hops":           q["num_hops"],
        "difficulty":         q["difficulty"],
        "predicted_answer":   answer,
        "success":            success,
        "wall_latency_ms":    round(wall_ms, 1),
        "trajectory":         TRAJECTORY_LOGGER.steps.copy(),
        "trajectory_summary": traj_sum,
        "D1_answer_correctness":     d1,
        "D2_retrieval_step_quality": d2,
        "D3_planning_coherence":     d3,
        "D5_trajectory_efficiency":  d5,
    }

print("Run function ready — scores D1, D2, D3, D5 per question")

Run function ready — scores D1, D2, D3, D5 per question


---
## Cell 6 — Run all 50 questions (incremental save)

Each question takes ~5–45s. Total ~8–12 minutes. Results saved after every question.

In [31]:
with open(PROJECT_ROOT / "data/questions/3_musique_50.json") as f:
    questions = json.load(f)

N_RUN = 50
to_run = questions[:N_RUN]
results = []

print(f"Running {N_RUN} questions — all 4 metrics (D1, D2, D3, D5)")
print("k=3 | system prompt ON | nomic-embed-text | max_steps=10")
print(f"recursion_limit=28 | timeout={QUESTION_TIMEOUT}s per question")
print("=" * 65)

for i, q in enumerate(to_run):
    print(f"\nQ{i+1}/{N_RUN}: {q['question'][:65]}...")
    print(f"  Expected: {q['answer']} | Hops: {q['num_hops']} | "
          f"RC={q['difficulty']['reasoning_complexity']} "
          f"RD={q['difficulty']['retrieval_difficulty']}")

    r = run_question(q)
    results.append(r)

    print(f"  Predicted: {r['predicted_answer'][:70]}")
    print(f"  D1={r['D1_answer_correctness']['score']:.3f}  "
          f"D2={r['D2_retrieval_step_quality']['d2_score']:.3f}  "
          f"D3={r['D3_planning_coherence']['d3_score']:.3f}  "
          f"D5={r['D5_trajectory_efficiency']['efficiency_score']:.3f}  "
          f"({r['wall_latency_ms']/1000:.0f}s)")

    # Save incrementally so you don't lose progress
    out_path = PROJECT_ROOT / "data/trajectories/3_agent_results.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)

print(f"\nSaved to {out_path}")

Running 50 questions — all 4 metrics (D1, D2, D3, D5)
k=3 | system prompt ON | nomic-embed-text | max_steps=10
recursion_limit=28 | timeout=120s per question

Q1/50: Who is the spouse of the Green performer?...
  Expected: Miquette Giraudy | Hops: 2 | RC=2 RD=3
  Predicted: Based on the searches, I found that Billie Joe Armstrong is the lead s
  D1=0.000  D2=0.467  D3=0.798  D5=0.903  (14s)

Q2/50: Who founded the company that distributed the film UHF?...
  Expected: Mike Medavoy | Hops: 2 | RC=2 RD=3
  Predicted: The founder of the company that distributed the film "UHF" is Kevin Sm
  D1=0.000  D2=0.000  D3=0.762  D5=0.882  (16s)

Q3/50: What administrative territorial entity is the owner of Ciudad Dep...
  Expected: Tamaulipas | Hops: 2 | RC=2 RD=3
  Predicted: Based on the searches, Ciudad Deportiva is located in Minsk Region or 
  D1=0.000  D2=0.000  D3=0.580  D5=0.920  (13s)

Q4/50: Where is Ulrich Walter's employer headquartered?...
  Expected: Cologne | Hops: 2 | RC=2 RD=3
  Pre

---
## Cell 7 — Full results table with all 4 dimensions

In [32]:
with open(PROJECT_ROOT / "data/trajectories/3_agent_results.json") as f:
    results = json.load(f)

n = len(results)

print("=" * 80)
print("WEEK 3 — RESULTS WITH ALL 4 METRICS")
print("=" * 80)
print(f"{'Q':3} {'Hops':5} {'RC':4} {'RD':4} {'Correct':8} {'D1':7} {'D2':7} {'D3':7} {'D5':7} {'Steps':7}")
print("-" * 80)

for i, r in enumerate(results):
    d = r['difficulty']
    print(
        f"{i+1:3} "
        f"{r['num_hops']:5} "
        f"{d['reasoning_complexity']:4} "
        f"{d['retrieval_difficulty']:4} "
        f"{'YES' if r['D1_answer_correctness']['exact_match'] else 'NO':8} "
        f"{r['D1_answer_correctness']['score']:7.3f} "
        f"{r['D2_retrieval_step_quality']['d2_score']:7.3f} "
        f"{r['D3_planning_coherence']['d3_score']:7.3f} "
        f"{r['D5_trajectory_efficiency']['efficiency_score']:7.3f} "
        f"{r['trajectory_summary'].get('total_steps',0):7}"
    )

print("-" * 80)

correct  = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
degenerate_count = sum(1 for r in results if r['D1_answer_correctness'].get('degenerate_output', False))
non_degen = [r for r in results if not r['D1_answer_correctness'].get('degenerate_output', False)]
n_nd = len(non_degen)

avg_d1   = sum(r['D1_answer_correctness']['score'] for r in results) / n
avg_d2   = sum(r['D2_retrieval_step_quality']['d2_score'] for r in non_degen) / n_nd if n_nd else 0
avg_d3   = sum(r['D3_planning_coherence']['d3_score'] for r in non_degen) / n_nd if n_nd else 0
avg_d5   = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in non_degen) / n_nd if n_nd else 0
avg_steps= sum(r['trajectory_summary'].get('total_steps',0) for r in non_degen) / n_nd if n_nd else 0
loops    = sum(1 for r in results if r['trajectory_summary'].get('loop_detected'))

# Tokens per correct answer (only average over correct, not inf/None)
correct_tokens = [
    r['D5_trajectory_efficiency']['tokens_per_correct_answer']
    for r in results
    if r['D1_answer_correctness']['exact_match']
    and r['D5_trajectory_efficiency']['tokens_per_correct_answer'] is not None
]
avg_tpca = sum(correct_tokens) / len(correct_tokens) if correct_tokens else 0

print(f"\nAGGREGATES ({n} questions, {degenerate_count} degenerate excluded from D2/D3/D5):")
print(f"  Accuracy (D1 exact):   {correct}/{n} = {correct/n:.0%}")
print(f"  Avg D1 score:          {avg_d1:.3f}")
print(f"  Avg D2 score:          {avg_d2:.3f}  ← retrieval step quality (excl. degenerate)")
print(f"  Avg D3 score:          {avg_d3:.3f}  ← planning coherence (excl. degenerate)")
print(f"  Avg D5 efficiency:     {avg_d5:.3f}  (excl. degenerate)")
print(f"  Avg steps/question:    {avg_steps:.1f}  (excl. degenerate)")
print(f"  Loops detected:        {loops}/{n}")
print(f"  Degenerate outputs:    {degenerate_count}/{n}")
print(f"  Tokens/correct answer: {avg_tpca:.1f}")

WEEK 3 — RESULTS WITH ALL 4 METRICS
Q   Hops  RC   RD   Correct  D1      D2      D3      D5      Steps  
--------------------------------------------------------------------------------
  1     2    2    3 NO         0.000   0.467   0.798   0.903       2
  2     2    2    3 NO         0.000   0.000   0.762   0.882       2
  3     2    2    3 NO         0.000   0.000   0.580   0.920       2
  4     2    2    3 NO         0.000   0.333   0.845   0.875       2
  5     2    2    3 YES        1.000   0.333   0.847   0.893       2
  6     2    2    3 NO         0.067   0.466   0.697   0.891       2
  7     2    2    3 NO         0.000   0.333   0.530   0.846       2
  8     2    2    3 YES        1.000   0.466   0.325   0.450       3
  9     2    2    3 NO         0.000   0.000   0.541   0.823       2
 10     2    2    3 NO         0.100   0.466   0.825   0.910       2
 11     2    2    3 NO         0.000   0.333   0.712   0.929       2
 12     2    2    3 NO         0.000   0.000   0.647   

---
## Cell 8 — D6 preview: difficulty interaction analysis

In [33]:
with open(PROJECT_ROOT / "data/trajectories/3_agent_results.json") as f:
    results = json.load(f)

# Split by number of high-difficulty axes
by_axes = {0:[], 1:[], 2:[], 3:[], 4:[]}
for r in results:
    n_axes = r['difficulty']['num_high_difficulty_axes']
    by_axes[n_axes].append(r)

print("=" * 55)
print("D6 PREVIEW — DIFFICULTY INTERACTION ANALYSIS")
print("=" * 55)
print(f"{'N high axes':15} {'Count':8} {'Accuracy':10} {'Avg D1':8}")
print("-" * 45)

for n_axes in sorted(by_axes.keys()):
    group = by_axes[n_axes]
    if not group:
        continue
    acc   = sum(1 for r in group if r['D1_answer_correctness']['exact_match'])
    avg_d1= sum(r['D1_answer_correctness']['score'] for r in group) / len(group)
    print(f"{n_axes:15} {len(group):8} {acc}/{len(group):>8}     {avg_d1:.3f}")

print()
print("Interpretation:")
print("  If accuracy drops as N high axes increases → D6 effect confirmed")
print("  Super-linear drop (0→1 axes: small drop, 1→2 axes: large drop)")
print("  is the key finding — multiple difficulty dimensions interact")
print("  and compound each other. RAGAS cannot detect this at all.")
print()
print("NOTE: With only 10 questions, this is a preview.")
print("The full 50-question dataset will give statistically meaningful D6 numbers.")
print("Week 7-9: run full benchmark on 50 questions for D6 analysis.")

D6 PREVIEW — DIFFICULTY INTERACTION ANALYSIS
N high axes     Count    Accuracy   Avg D1  
---------------------------------------------
              2       50 11/      50     0.267

Interpretation:
  If accuracy drops as N high axes increases → D6 effect confirmed
  Super-linear drop (0→1 axes: small drop, 1→2 axes: large drop)
  is the key finding — multiple difficulty dimensions interact
  and compound each other. RAGAS cannot detect this at all.

NOTE: With only 10 questions, this is a preview.
The full 50-question dataset will give statistically meaningful D6 numbers.
Week 7-9: run full benchmark on 50 questions for D6 analysis.


---
## Cell 9 — Push dataset v1 to HuggingFace

**Prerequisites:** HuggingFace account + `HF_TOKEN` in your `.env` file

In [34]:
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN not found in .env")
    print("To push to HuggingFace:")
    print("  1. Go to huggingface.co/settings/tokens")
    print("  2. Create a token with write permissions")
    print("  3. Add HF_TOKEN=your_token to your .env file")
    print("  4. Re-run this cell")
    print()
    print("Skipping HuggingFace push. Dataset saved locally at:")
    print(f"  {PROJECT_ROOT}/data/questions/3_musique_50.json")
else:
    from datasets import Dataset as HFDataset
    from huggingface_hub import login

    login(token=hf_token)

    with open(PROJECT_ROOT / "data/questions/3_musique_50.json") as f:
        questions = json.load(f)

    # Convert to HuggingFace Dataset format
    # Drop the full paragraphs list to keep the HF dataset lightweight
    # The full paragraphs are in the local JSON
    hf_data = {
        "id":                       [q["id"] for q in questions],
        "question":                 [q["question"] for q in questions],
        "answer":                   [q["answer"] for q in questions],
        "num_hops":                 [q["num_hops"] for q in questions],
        "num_supporting":           [q["num_supporting"] for q in questions],
        "num_distractors":          [q["num_distractors"] for q in questions],
        "reasoning_complexity":     [q["difficulty"]["reasoning_complexity"] for q in questions],
        "retrieval_difficulty":     [q["difficulty"]["retrieval_difficulty"] for q in questions],
        "source_structure":         [q["difficulty"]["source_structure"] for q in questions],
        "explainability_required":  [q["difficulty"]["explainability_required"] for q in questions],
        "multi_axis_hard":          [q["difficulty"]["multi_axis_hard"] for q in questions],
        "supporting_paragraphs":    [q["supporting_paragraphs"] for q in questions],
        "all_paragraphs":           [q["paragraphs"] for q in questions],
    }

    hf_dataset = HFDataset.from_dict(hf_data)

    # Push — replace with your actual HuggingFace username
    REPO_NAME = "agenticrag-bench-v1"  # will be at: huggingface.co/datasets/YOUR_NAME/agenticrag-bench-v1

    hf_dataset.push_to_hub(
        REPO_NAME,
        token=hf_token,
        commit_message="Dataset v1: 50 MuSiQue questions with 4-axis difficulty labels"
    )

    print(f"Dataset pushed to HuggingFace!")
    print(f"Load it anywhere with:")
    print(f'  from datasets import load_dataset')
    print(f'  ds = load_dataset("YOUR_HF_USERNAME/{REPO_NAME}")')

HF_TOKEN not found in .env
To push to HuggingFace:
  1. Go to huggingface.co/settings/tokens
  2. Create a token with write permissions
  3. Add HF_TOKEN=your_token to your .env file
  4. Re-run this cell

Skipping HuggingFace push. Dataset saved locally at:
  /Users/idhantsingh/Desktop/agenticrag-bench/data/questions/3_musique_50.json


---
## Cell 10 — Re-score Week 2 results with D2 and D3

In [35]:
week2_path = PROJECT_ROOT / "data/trajectories/2_agent_results.json"
week2_q_path = PROJECT_ROOT / "data/questions/2_musique_10.json"

if week2_path.exists() and week2_q_path.exists():
    with open(week2_path) as f:
        w2_results = json.load(f)
    with open(week2_q_path) as f:
        w2_questions = json.load(f)

    # Build question lookup for supporting paragraphs
    q_lookup = {q['id']: q for q in w2_questions}

    print("Re-scoring Week 2 results with D2 and D3...")
    print()

    for r in w2_results:
        q = q_lookup.get(r['question_id'], {})
        supporting = q.get('supporting_paragraphs', [])

        # Add D2
        r['D2_retrieval_step_quality'] = d2_metric.score(
            r.get('trajectory', []), supporting
        )
        # Add D3
        r['D3_planning_coherence'] = d3_metric.score(
            r.get('trajectory_summary', {}), r.get('num_hops', 2)
        )

    # Save
    out = PROJECT_ROOT / "data/trajectories/2_agent_results_full_metrics.json"
    with open(out, "w") as f:
        json.dump(w2_results, f, indent=2)

    print("Week 2 results with D2+D3 added:")
    print(f"{'Q':3} {'D1':7} {'D2':7} {'D3':7} {'D5':7}")
    print("-" * 35)
    for i, r in enumerate(w2_results):
        print(
            f"{i+1:3} "
            f"{r['D1_answer_correctness']['score']:7.3f} "
            f"{r['D2_retrieval_step_quality']['d2_score']:7.3f} "
            f"{r['D3_planning_coherence']['d3_score']:7.3f} "
            f"{r['D5_trajectory_efficiency']['efficiency_score']:7.3f}"
        )
    print(f"\nSaved to {out}")
else:
    print("Week 2 results not found — skipping re-scoring")

Re-scoring Week 2 results with D2 and D3...

Week 2 results with D2+D3 added:
Q   D1      D2      D3      D5     
-----------------------------------
  1   0.000   0.360   0.861   0.873
  2   0.100   0.360   0.842   0.827
  3   0.000   0.280   0.857   0.782
  4   1.000   0.280   0.703   0.739
  5   1.000   0.280   0.897   0.825

Saved to /Users/idhantsingh/Desktop/agenticrag-bench/data/trajectories/2_agent_results_full_metrics.json


---
## Cell 11 — Week 3 summary

In [36]:
with open(PROJECT_ROOT / "data/trajectories/3_agent_results.json") as f:
    results = json.load(f)

n       = len(results)
correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
degen   = sum(1 for r in results if r['D1_answer_correctness'].get('degenerate_output', False))
non_degen = [r for r in results if not r['D1_answer_correctness'].get('degenerate_output', False)]
n_nd = len(non_degen)

avg_d1  = sum(r['D1_answer_correctness']['score'] for r in results) / n
avg_d2  = sum(r['D2_retrieval_step_quality']['d2_score'] for r in non_degen) / n_nd if n_nd else 0
avg_d3  = sum(r['D3_planning_coherence']['d3_score'] for r in non_degen) / n_nd if n_nd else 0
avg_d5  = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in non_degen) / n_nd if n_nd else 0
avg_steps = sum(r['trajectory_summary'].get('total_steps',0) for r in non_degen) / n_nd if n_nd else 0
loops   = sum(1 for r in results if r['trajectory_summary'].get('loop_detected'))

correct_runs = [r for r in results if r['D1_answer_correctness']['exact_match']]
incorrect_runs = [r for r in non_degen if not r['D1_answer_correctness']['exact_match']]
corr_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in correct_runs)/len(correct_runs) if correct_runs else 0
corr_d3 = sum(r['D3_planning_coherence']['d3_score'] for r in correct_runs)/len(correct_runs) if correct_runs else 0
incorr_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in incorrect_runs)/len(incorrect_runs) if incorrect_runs else 0
incorr_d3 = sum(r['D3_planning_coherence']['d3_score'] for r in incorrect_runs)/len(incorrect_runs) if incorrect_runs else 0

correct_tokens = [
    r['D5_trajectory_efficiency']['tokens_per_correct_answer']
    for r in results
    if r['D1_answer_correctness']['exact_match']
    and r['D5_trajectory_efficiency']['tokens_per_correct_answer'] is not None
]
avg_tpca = sum(correct_tokens) / len(correct_tokens) if correct_tokens else 0

notes = f"""# Week 3 Observations — AgenticRAG-Bench
Date: {time.strftime('%Y-%m-%d')}

## What was built
- Dataset v1: 50 questions with 4-axis difficulty labels
- D2 (retrieval step quality) implemented and validated
- D3 (planning coherence) implemented and validated
- All 4 metrics run on {n} questions
- Week 2 results retroactively scored with D2 and D3
- D6 difficulty interaction preview analysis
- Degenerate output detection added to D1
- Undershooting nudge added to search tool

## Results ({n} questions, all 4 metrics)
- Accuracy:       {correct}/{n} = {correct/n:.0%}
- Avg D1 score:   {avg_d1:.3f}
- Avg D2 score:   {avg_d2:.3f}  (retrieval step quality, excl. {degen} degenerate)
- Avg D3 score:   {avg_d3:.3f}  (planning coherence, excl. {degen} degenerate)
- Avg D5 score:   {avg_d5:.3f}  (trajectory efficiency, excl. {degen} degenerate)
- Avg steps/q:    {avg_steps:.2f}
- Loops detected: {loops}/{n}
- Degenerate:     {degen}/{n}
- Tokens/correct: {avg_tpca:.1f}

## Key observations
- **D2 vs D1:** Even at {correct/n:.0%} accuracy, retrieval finds some correct context. Correct answers had D2={corr_d2:.3f} vs incorrect D2={incorr_d2:.3f} — bad retrieval strongly drives D1 failure.
- **D3 undershooting:** Agent averages {avg_steps:.2f} steps for 2-hop questions. Correct answers had D3={corr_d3:.3f} vs incorrect D3={incorr_d3:.3f}. Zero loops detected — the agent quits early, not loops.
- **D2/D3 correlate with D1:** Both metrics are ~2x higher for correct vs incorrect answers. Confirms diagnostic dimensions measure real signal.
- **D6 preview:** All {n} questions fell into the same difficulty bucket (RC=2, RD=3). Need single-hop baselines for a real interaction curve.

## Bugs fixed
- float('inf') in tokens_per_correct_answer → None (JSON safety)
- Degenerate output detection (raw tool dumps, refusals)
- D5 returns 0.0 for degenerate outputs (was inflating average)
- Undershooting nudge injected after first search step
- D2/D3 averages exclude degenerate questions

## Week 4 plan
- Implement D4 (noise robustness + interference rate)
- Add controlled noise injection to 25% of questions
- Run evaluation with and without noise — measure interference rate
- Test query rewriting to improve D2 on the 24 zero-retrieval questions
"""

with open(PROJECT_ROOT / "notes/3_observations.md", "w") as f:
    f.write(notes)

print("=" * 65)
print("WEEK 3 COMPLETE")
print("=" * 65)
print()
print("Files created:")
print(f"  data/questions/3_musique_50.json          <- dataset v1 ({n} questions)")
print(f"  data/trajectories/3_agent_results.json    <- {n} questions, 4 metrics")
print("  data/trajectories/2_agent_results_full_metrics.json <- week2 retroactively scored")
print("  notes/3_observations.md                    <- auto-generated observations")
print()
print("Commit:")
print("  git add .")
print("  git commit -m 'week3: D2+D3 metrics, 50-question dataset, D6 preview'")
print("  git push")

WEEK 3 COMPLETE

Files created:
  data/questions/3_musique_50.json          <- dataset v1 (50 questions)
  data/trajectories/3_agent_results.json    <- 50 questions, 4 metrics
  data/trajectories/2_agent_results_full_metrics.json <- week2 retroactively scored
  notes/3_observations.md                    <- auto-generated observations

Commit:
  git add .
  git commit -m 'week3: D2+D3 metrics, 50-question dataset, D6 preview'
  git push
